# Ñ · Fase A1: el ejecutable con tipos, en Linux

Corre en CPU las suites que usan el ejecutable de `n-compilar` (`exp-aot` completa con `N_AOT_COMPROBAR=1`,
`exp-lengua-franca`, `exp-sin-comprobar`, `exp-interop`) y mide `velocidad/bench.esl` en la VM y como ejecutable.
El proyecto sale del dataset `n-motor-exp` (la rama `motor-exp` con A1, sin commit).


In [ ]:
# 1 · Node y el proyecto (del dataset)
import os, subprocess, shutil, re, glob, json, time
TRABAJO = '/kaggle/temp/n-etapas'; SALIDA = '/kaggle/working'
os.makedirs(TRABAJO, exist_ok=True)
def version_node():
    try: return subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip()
    except FileNotFoundError: return ''
v = version_node()
if not v or int(re.match(r'v(\d+)', v).group(1)) < 18:
    NODE = 'v22.12.0'
    subprocess.run(f'curl -fsSL https://nodejs.org/dist/{NODE}/node-{NODE}-linux-x64.tar.xz | tar -xJ -C {TRABAJO}', shell=True, check=True)
    os.environ['PATH'] = f'{TRABAJO}/node-{NODE}-linux-x64/bin:' + os.environ['PATH']
print('node', version_node())
PROYECTO = f'{TRABAJO}/n'
candidatos = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/motor/rutas.js', recursive=True)]
if not candidatos: raise SystemExit('no encuentro el proyecto en /kaggle/input: añade el dataset n-motor-exp')
if os.path.exists(PROYECTO): shutil.rmtree(PROYECTO)
shutil.copytree(candidatos[0], PROYECTO, ignore=shutil.ignore_patterns('node_modules')); print('proyecto copiado desde', candidatos[0])
os.chdir(PROYECTO)
subprocess.run('npm install koffi --no-audit --no-fund --loglevel=error', shell=True, check=True, timeout=900)
print(subprocess.run('node motor-exp/util/construir.js', shell=True, capture_output=True, text=True).stdout.strip())
print(subprocess.run('nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv', shell=True, capture_output=True, text=True).stdout.strip())
print(subprocess.run('gcc --version | head -1; g++ --version | head -1; python3 -c "import numpy; print(\'numpy\', numpy.__version__)"', shell=True, capture_output=True, text=True).stdout.strip())

In [ ]:
# 2 · Las suites del ejecutable, con las comprobaciones de tipos encendidas
import os, subprocess, time, json
assert 'n_hilo.profundidad = 0;' in open(f'{PROYECTO}/motor-exp/aot/nrt_biblioteca.inc', encoding='utf-8').read(), 'el dataset es el viejo (sin el arreglo de A2 para los hilos): espera a que Kaggle procese la version nueva'
os.environ['N_AOT_COMPROBAR'] = '1'
SUITES = [
    ('A1 · exp-aot completa (N_AOT_COMPROBAR=1)', 'exp-aot.js --ver'),
    ('N4.6 · lengua franca',                      'exp-lengua-franca.js'),
    ('N4.11 · sinComprobar',                      'exp-sin-comprobar.js'),
    ('N4.10 · interop',                           'exp-interop.js'),
]
resultados = []
for nombre, orden in SUITES:
    t0 = time.time()
    try:
        r = subprocess.run(f'node motor-exp/pruebas/{orden}', shell=True, capture_output=True, text=True, cwd=PROYECTO, timeout=3600)
        salida, codigo = r.stdout + r.stderr, r.returncode
    except subprocess.TimeoutExpired as e:
        salida, codigo = (e.stdout or '') + (e.stderr or '') + chr(10) + '(se paso de 60 minutos)', 'tiempo'
    lineas = [l for l in salida.splitlines() if l.strip()]
    resumen = lineas[-1].strip() if lineas else 'sin salida'
    fallos = [l.strip() for l in lineas if l.lstrip().startswith('✗')]
    resultados.append(dict(etapa=nombre, orden=orden, codigo=codigo, segundos=round(time.time() - t0), resumen=resumen, fallos=fallos))
    print(f'{"✓" if codigo == 0 else "✗"} {nombre:44} {resumen}  ({round(time.time() - t0)} s)')
    for f in fallos[:12]: print('      ' + f)
    with open(f'{SALIDA}/salida-{orden.split()[0]}.txt', 'w') as f: f.write(salida)
json.dump(resultados, open(f'{SALIDA}/a1-resultados.json', 'w'), ensure_ascii=False, indent=1)
print()
print('en verde:', sum(r['codigo'] == 0 for r in resultados), 'de', len(resultados))


In [ ]:
# 3 · velocidad/bench.esl: la VM contra el ejecutable (sin comprobaciones), mejor de 3
os.environ.pop('N_AOT_COMPROBAR', None)
def corre(orden):
    mejor = {}
    for _ in range(3):
        out = subprocess.run(orden, shell=True, capture_output=True, text=True, cwd=PROYECTO, timeout=600).stdout
        for l in out.splitlines():
            if ' en ' in l and l.endswith(' ms'):
                k, ms = l.split(' = ')[0], int(l.rsplit(' en ', 1)[1][:-3])
                mejor[k] = min(mejor.get(k, 10**9), ms)
    return mejor
print('VM        ', corre('node motor-exp/util/n-node.js velocidad/bench.esl'))
print(subprocess.run('node motor-exp/util/n-compilar.js velocidad/bench.esl -o /kaggle/working/bench-n', shell=True, capture_output=True, text=True, cwd=PROYECTO).stdout.strip())
print('ejecutable', corre('/kaggle/working/bench-n'))
print(subprocess.run('gcc -O2 -o /kaggle/working/bench-c velocidad/bench.c -lm', shell=True, capture_output=True, text=True, cwd=PROYECTO).stderr.strip())
print('C -O2     ', corre('/kaggle/working/bench-c'))
